In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyxdf

In [3]:
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

XDF_PATH = RAW_DIR / "session_01.xdf"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("XDF_PATH:", XDF_PATH)
print("Exists:", XDF_PATH.exists())

PROJECT_ROOT: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony
XDF_PATH: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/raw/session_01.xdf
Exists: True


In [4]:
streams, header = pyxdf.load_xdf(str(XDF_PATH))
print(f"Loaded {len(streams)} streams")

Loaded 22 streams


In [5]:
def safe_get(obj, *keys, default=None):
    cur = obj
    try:
        for k in keys:
            cur = cur[k]
        return cur
    except (KeyError, IndexError, TypeError):
        return default


def to_str(x):
    if x is None:
        return None
    if isinstance(x, bytes):
        return x.decode("utf-8", errors="replace")
    return str(x)


def clean_marker_text(x):
    if x is None:
        return None
    return str(x).strip().lower()


def extract_emotibit_id(stream_name):
    # Example: PPG_EmotiBit_4 -> EmotiBit_4
    if stream_name is None:
        return None
    parts = str(stream_name).split("PPG_")
    if len(parts) == 2:
        return parts[1]
    return str(stream_name)


def build_stream_inventory(streams):
    rows = []
    for idx, stream in enumerate(streams):
        rows.append({
            "stream_index": idx,
            "name": to_str(safe_get(stream, "info", "name", 0)),
            "type": to_str(safe_get(stream, "info", "type", 0)),
            "channel_count": safe_get(stream, "info", "channel_count", 0),
            "nominal_srate": safe_get(stream, "info", "nominal_srate", 0),
            "n_samples": len(stream.get("time_stamps", [])),
            "time_series_shape": np.asarray(stream.get("time_series", [])).shape,
        })
    return pd.DataFrame(rows)

Stream inventory and PPG stream selection

In [6]:
stream_inventory = build_stream_inventory(streams)
display(stream_inventory)

ppg_streams = stream_inventory[
    (stream_inventory["type"].str.lower() == "ppg") |
    (stream_inventory["name"].str.contains("PPG_EmotiBit_", case=False, na=False))
].copy()

ppg_streams = ppg_streams.sort_values("name").reset_index(drop=True)
display(ppg_streams)

,stream_index,name,type,channel_count,nominal_srate,n_samples,time_series_shape
0,0,EDA_EmotiBit_1,EDA,1,15.00000000000000,16893,"(16893, 1)"
1,1,EDA_EmotiBit_3,EDA,1,15.00000000000000,16817,"(16817, 1)"
2,2,TEMP_EmotiBit_7,TEMP,1,15.00000000000000,16931,"(16931, 1)"
3,3,TEMP_EmotiBit_6,TEMP,1,15.00000000000000,16810,"(16810, 1)"
4,4,PPG_EmotiBit_4,PPG,3,100.0000000000000,111383,"(111383, 3)"
5,5,EDA_EmotiBit_6,EDA,1,15.00000000000000,16810,"(16810, 1)"
6,6,TEMP_EmotiBit_1,TEMP,1,15.00000000000000,16893,"(16893, 1)"
7,7,MarkerStream,Markers,1,0.000000000000000,7,"(7, 1)"
8,8,EDA_EmotiBit_4,EDA,1,15.00000000000000,16809,"(16809, 1)"
9,9,PPG_EmotiBit_1,PPG,3,100.0000000000000,111837,"(111837, 3)"


,stream_index,name,type,channel_count,nominal_srate,n_samples,time_series_shape
0,9,PPG_EmotiBit_1,PPG,3,100.0000000000000,111837,"(111837, 3)"
1,20,PPG_EmotiBit_2,PPG,3,100.0000000000000,112432,"(112432, 3)"
2,19,PPG_EmotiBit_3,PPG,3,100.0000000000000,111168,"(111168, 3)"
3,4,PPG_EmotiBit_4,PPG,3,100.0000000000000,111383,"(111383, 3)"
4,17,PPG_EmotiBit_5,PPG,3,100.0000000000000,111277,"(111277, 3)"
5,15,PPG_EmotiBit_6,PPG,3,100.0000000000000,111362,"(111362, 3)"
6,14,PPG_EmotiBit_7,PPG,3,100.0000000000000,112387,"(112387, 3)"


In [7]:
MARKER_STREAM_INDEX = 7

marker_stream = streams[MARKER_STREAM_INDEX]
marker_rows = []

for ts, val in zip(marker_stream["time_stamps"], marker_stream["time_series"]):
    if isinstance(val, (list, np.ndarray)) and len(val) > 0:
        raw_text = to_str(val[0])
    else:
        raw_text = to_str(val)

    marker_rows.append({
        "timestamp": float(ts),
        "marker_raw": raw_text,
        "marker_clean": clean_marker_text(raw_text),
    })

marker_timeline = pd.DataFrame(marker_rows).sort_values("timestamp").reset_index(drop=True)
display(marker_timeline)

,timestamp,marker_raw,marker_clean
0,62858.874887,baseline,baseline
1,63006.653002,start,start
2,63102.985040,longer version meditation,longer version meditation
3,63564.003053,inbetween marker,inbetween marker
4,63707.023919,quiet session,quiet session
5,63856.455028,eyes open,eyes open
6,63951.232376,stop,stop


In [8]:
def first_timestamp_for_marker(df, marker_name):
    matches = df.loc[df["marker_clean"] == marker_name, "timestamp"]
    if len(matches) == 0:
        return np.nan
    return float(matches.iloc[0])

segment_defs = [
    ("pre_meditation", "baseline", "longer version meditation"),
    ("main_meditation", "longer version meditation", "eyes open"),
    ("recovery", "eyes open", "stop"),
]

segment_rows = []
for segment_name, start_marker, end_marker in segment_defs:
    start_ts = first_timestamp_for_marker(marker_timeline, start_marker)
    end_ts = first_timestamp_for_marker(marker_timeline, end_marker)

    segment_rows.append({
        "segment": segment_name,
        "start_marker": start_marker,
        "end_marker": end_marker,
        "start_time": start_ts,
        "end_time": end_ts,
        "duration_sec": end_ts - start_ts if pd.notna(start_ts) and pd.notna(end_ts) else np.nan,
        "valid": pd.notna(start_ts) and pd.notna(end_ts) and end_ts > start_ts,
    })

segments_df = pd.DataFrame(segment_rows)
display(segments_df)

,segment,start_marker,end_marker,start_time,end_time,duration_sec,valid
0,pre_meditation,baseline,longer version meditation,62858.874887,63102.985040,244.110153,True
1,main_meditation,longer version meditation,eyes open,63102.985040,63856.455028,753.469988,True
2,recovery,eyes open,stop,63856.455028,63951.232376,94.777348,True


In [9]:
segments_path = TABLES_DIR / "segment_definitions.csv"
segments_df.to_csv(segments_path, index=False)
print("Saved:", segments_path)

Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/outputs/tables/segment_definitions.csv


In [10]:
ppg_summary_rows = []

for _, row in ppg_streams.iterrows():
    stream_index = int(row["stream_index"])
    stream = streams[stream_index]

    stream_name = to_str(safe_get(stream, "info", "name", 0))
    emotibit_id = extract_emotibit_id(stream_name)

    timestamps = np.asarray(stream["time_stamps"], dtype=float)
    values = np.asarray(stream["time_series"], dtype=float)

    if values.ndim != 2 or values.shape[1] < 3:
        print(f"Skipping {stream_name}: unexpected shape {values.shape}")
        continue

    df = pd.DataFrame({
        "timestamp": timestamps,
        "time_from_start_sec": timestamps - timestamps[0],
        "PPG_1": values[:, 0],
        "PPG_2": values[:, 1],
        "PPG_3": values[:, 2],
    })

    out_path = PROCESSED_DIR / f"{emotibit_id}_full.csv"
    df.to_csv(out_path, index=False)

    ppg_summary_rows.append({
        "stream_index": stream_index,
        "stream_name": stream_name,
        "emotibit_id": emotibit_id,
        "n_samples": len(df),
        "start_time": float(df["timestamp"].iloc[0]),
        "end_time": float(df["timestamp"].iloc[-1]),
        "duration_sec": float(df["timestamp"].iloc[-1] - df["timestamp"].iloc[0]),
        "output_csv": out_path.name,
    })

    print("Saved:", out_path)

ppg_extraction_summary = pd.DataFrame(ppg_summary_rows).sort_values("emotibit_id").reset_index(drop=True)
display(ppg_extraction_summary)

Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_1_full.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_2_full.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_3_full.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_4_full.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_5_full.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_6_full.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_7_full.csv


,stream_index,stream_name,emotibit_id,n_samples,start_time,end_time,duration_sec,output_csv
0,9,PPG_EmotiBit_1,EmotiBit_1,111837,62847.559699,63971.584144,1124.024445,EmotiBit_1_full.csv
1,20,PPG_EmotiBit_2,EmotiBit_2,112432,62847.766658,63971.842057,1124.075399,EmotiBit_2_full.csv
2,19,PPG_EmotiBit_3,EmotiBit_3,111168,62847.570262,63971.375708,1123.805446,EmotiBit_3_full.csv
3,4,PPG_EmotiBit_4,EmotiBit_4,111383,62847.050907,63971.326119,1124.275212,EmotiBit_4_full.csv
4,17,PPG_EmotiBit_5,EmotiBit_5,111277,62847.424697,63971.290235,1123.865539,EmotiBit_5_full.csv
5,15,PPG_EmotiBit_6,EmotiBit_6,111362,62846.939475,63970.456990,1123.517515,EmotiBit_6_full.csv
6,14,PPG_EmotiBit_7,EmotiBit_7,112387,62847.365914,63971.537217,1124.171303,EmotiBit_7_full.csv


In [11]:
summary_path = TABLES_DIR / "ppg_extraction_summary.csv"
ppg_extraction_summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)

Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/outputs/tables/ppg_extraction_summary.csv


Export segmented (based on markers) CSVs for each EmotiBit

In [12]:
segment_export_rows = []

for _, row in ppg_streams.iterrows():
    stream_index = int(row["stream_index"])
    stream = streams[stream_index]

    stream_name = to_str(safe_get(stream, "info", "name", 0))
    emotibit_id = extract_emotibit_id(stream_name)

    timestamps = np.asarray(stream["time_stamps"], dtype=float)
    values = np.asarray(stream["time_series"], dtype=float)

    if values.ndim != 2 or values.shape[1] < 3:
        print(f"Skipping {stream_name}: unexpected shape {values.shape}")
        continue

    full_df = pd.DataFrame({
        "timestamp": timestamps,
        "PPG_1": values[:, 0],
        "PPG_2": values[:, 1],
        "PPG_3": values[:, 2],
    })

    for _, seg in segments_df.iterrows():
        segment_name = seg["segment"]
        start_time = seg["start_time"]
        end_time = seg["end_time"]
        valid = seg["valid"]

        if not valid:
            continue

        seg_df = full_df[(full_df["timestamp"] >= start_time) & (full_df["timestamp"] <= end_time)].copy()

        if len(seg_df) == 0:
            print(f"No data for {emotibit_id} in {segment_name}")
            continue

        seg_df["time_from_segment_start_sec"] = seg_df["timestamp"] - seg_df["timestamp"].iloc[0]

        out_path = PROCESSED_DIR / f"{emotibit_id}_{segment_name}.csv"
        seg_df.to_csv(out_path, index=False)

        segment_export_rows.append({
            "emotibit_id": emotibit_id,
            "segment": segment_name,
            "start_time": float(seg_df["timestamp"].iloc[0]),
            "end_time": float(seg_df["timestamp"].iloc[-1]),
            "n_samples": len(seg_df),
            "duration_sec": float(seg_df["timestamp"].iloc[-1] - seg_df["timestamp"].iloc[0]),
            "output_csv": out_path.name,
        })

        print("Saved:", out_path)

segment_export_summary = pd.DataFrame(segment_export_rows).sort_values(
    ["emotibit_id", "segment"]
).reset_index(drop=True)

display(segment_export_summary)

Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_1_pre_meditation.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_1_main_meditation.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_1_recovery.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_2_pre_meditation.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_2_main_meditation.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_2_recovery.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_3_pre_meditation.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_3_main_meditation.csv
Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/data/processed/EmotiBit_3_re

,emotibit_id,segment,start_time,end_time,n_samples,duration_sec,output_csv
0,EmotiBit_1,main_meditation,63102.986907,63856.453954,74968,753.467046,EmotiBit_1_main_meditation.csv
1,EmotiBit_1,pre_meditation,62858.876731,63102.976857,24288,244.100126,EmotiBit_1_pre_meditation.csv
2,EmotiBit_1,recovery,63856.464004,63951.231579,9430,94.767575,EmotiBit_1_recovery.csv
3,EmotiBit_2,main_meditation,63102.993390,63856.446140,75362,753.452750,EmotiBit_2_main_meditation.csv
4,EmotiBit_2,pre_meditation,62858.884337,63102.983392,24416,244.099055,EmotiBit_2_pre_meditation.csv
5,EmotiBit_2,recovery,63856.456138,63951.226359,9480,94.770221,EmotiBit_2_recovery.csv
6,EmotiBit_3,main_meditation,63102.988403,63856.454731,74534,753.466328,EmotiBit_3_main_meditation.csv
7,EmotiBit_3,pre_meditation,62858.882417,63102.978293,24147,244.095877,EmotiBit_3_pre_meditation.csv
8,EmotiBit_3,recovery,63856.464840,63951.228144,9375,94.763304,EmotiBit_3_recovery.csv
9,EmotiBit_4,main_meditation,63102.991025,63856.447806,74646,753.456781,EmotiBit_4_main_meditation.csv


In [13]:
segment_summary_path = TABLES_DIR / "ppg_segment_export_summary.csv"
segment_export_summary.to_csv(segment_summary_path, index=False)
print("Saved:", segment_summary_path)

Saved: /Users/sabirthapa/Desktop/emotibit/meditation_group_synchrony/outputs/tables/ppg_segment_export_summary.csv


In [14]:
print("=== QUICK VALIDATION ===")
print("Number of PPG streams found:", len(ppg_streams))
print("Expected: 7")

print("\nSegments:")
display(segments_df)

print("\nExtraction summary:")
display(ppg_extraction_summary)

print("\nSegment export summary:")
display(segment_export_summary.head(20))

=== QUICK VALIDATION ===
Number of PPG streams found: 7
Expected: 7

Segments:


,segment,start_marker,end_marker,start_time,end_time,duration_sec,valid
0,pre_meditation,baseline,longer version meditation,62858.874887,63102.985040,244.110153,True
1,main_meditation,longer version meditation,eyes open,63102.985040,63856.455028,753.469988,True
2,recovery,eyes open,stop,63856.455028,63951.232376,94.777348,True



Extraction summary:


,stream_index,stream_name,emotibit_id,n_samples,start_time,end_time,duration_sec,output_csv
0,9,PPG_EmotiBit_1,EmotiBit_1,111837,62847.559699,63971.584144,1124.024445,EmotiBit_1_full.csv
1,20,PPG_EmotiBit_2,EmotiBit_2,112432,62847.766658,63971.842057,1124.075399,EmotiBit_2_full.csv
2,19,PPG_EmotiBit_3,EmotiBit_3,111168,62847.570262,63971.375708,1123.805446,EmotiBit_3_full.csv
3,4,PPG_EmotiBit_4,EmotiBit_4,111383,62847.050907,63971.326119,1124.275212,EmotiBit_4_full.csv
4,17,PPG_EmotiBit_5,EmotiBit_5,111277,62847.424697,63971.290235,1123.865539,EmotiBit_5_full.csv
5,15,PPG_EmotiBit_6,EmotiBit_6,111362,62846.939475,63970.456990,1123.517515,EmotiBit_6_full.csv
6,14,PPG_EmotiBit_7,EmotiBit_7,112387,62847.365914,63971.537217,1124.171303,EmotiBit_7_full.csv



Segment export summary:


,emotibit_id,segment,start_time,end_time,n_samples,duration_sec,output_csv
0,EmotiBit_1,main_meditation,63102.986907,63856.453954,74968,753.467046,EmotiBit_1_main_meditation.csv
1,EmotiBit_1,pre_meditation,62858.876731,63102.976857,24288,244.100126,EmotiBit_1_pre_meditation.csv
2,EmotiBit_1,recovery,63856.464004,63951.231579,9430,94.767575,EmotiBit_1_recovery.csv
3,EmotiBit_2,main_meditation,63102.993390,63856.446140,75362,753.452750,EmotiBit_2_main_meditation.csv
4,EmotiBit_2,pre_meditation,62858.884337,63102.983392,24416,244.099055,EmotiBit_2_pre_meditation.csv
5,EmotiBit_2,recovery,63856.456138,63951.226359,9480,94.770221,EmotiBit_2_recovery.csv
6,EmotiBit_3,main_meditation,63102.988403,63856.454731,74534,753.466328,EmotiBit_3_main_meditation.csv
7,EmotiBit_3,pre_meditation,62858.882417,63102.978293,24147,244.095877,EmotiBit_3_pre_meditation.csv
8,EmotiBit_3,recovery,63856.464840,63951.228144,9375,94.763304,EmotiBit_3_recovery.csv
9,EmotiBit_4,main_meditation,63102.991025,63856.447806,74646,753.456781,EmotiBit_4_main_meditation.csv
